# DTLS — CoAP over coaps://

A greenhouse node (`CoapServer`) and a client talk CoAP through DTLS 1.2 on an in-memory datagram network: first with a pre-shared key, then with a pinned certificate. For a real device replace `net.Bind(...)` with `new UdpDatagramTransport(...)` and use port 5684.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.28.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.28.0-preview.1"
#r "nuget: IoTCom.Net.Transport.Dtls, 0.28.0-preview.1"

## Pre-shared key
The RFC 7925 profile for constrained devices: TLS_PSK_WITH_AES_128_CCM_8. The server looks keys up by identity; keys never appear in logs or `ToString()`.

In [ ]:
using System.Net;
using IoTCom.Net;
using IoTCom.Net.Protocols.Coap;
using IoTCom.Net.Transport.Dtls;
using IoTCom.Net.Transports;

var net = new InMemoryDatagramNetwork();
var at = new IPEndPoint(IPAddress.Parse("10.0.0.50"), 5684);
var psk = DtlsPsk.FromHex("greenhouse-node-01", "6A2F0C9B1D4E7F3A8B5C2D1E0F9A8B7C");
var dtlsServer = new DtlsServerTransport(net.Bind(at), new DtlsServerOptions().UsePsk(psk));
dtlsServer.SessionEstablished += s => Console.WriteLine($"server: session with {s.Remote}, {s.CipherSuite}, identity {s.PskIdentity}");
var node = CoapServer.Create(o => o.UseDatagramTransport(() => dtlsServer));
var device = new CoapDeviceSimulator(node);
await node.StartAsync();

var client = CoapClient.Create(o => o.UseDtlsClient(() => net.Bind(), at, new DtlsClientOptions { Psk = psk }).UseServer(at));
await client.ConnectAsync();
Console.WriteLine($"temperature {(await client.GetAsync("/sensors/temperature")).PayloadText} °C  (key: {psk})");

## A wrong key
Records with a bad MAC are dropped silently (RFC 6347), so a wrong key ends in a handshake timeout with a hint.

In [ ]:
var wrong = new DtlsClientTransport(net.Bind(), at, new DtlsClientOptions { Psk = DtlsPsk.FromHex("greenhouse-node-01", "00112233445566778899AABBCCDDEEFF"), HandshakeTimeout = TimeSpan.FromSeconds(2) });
try { await wrong.HandshakeAsync(); } catch (IoTComException e) { Console.WriteLine(e.Message); }
await wrong.DisposeAsync();
await client.DisposeAsync();
await device.DisposeAsync();
await node.DisposeAsync();

## Certificates and pinning
ECDHE_ECDSA with a self-signed P-256 certificate; the client accepts exactly the fingerprint it pins.

In [ ]:
var cert = DtlsCertificates.CreateSelfSigned("greenhouse-gw.local");
var gw = CoapServer.Create(o => o.UseDtlsServer(() => net.Bind(new IPEndPoint(IPAddress.Parse("10.0.0.51"), 5684)), new DtlsServerOptions { Certificate = cert }));
var gwDevice = new CoapDeviceSimulator(gw);
await gw.StartAsync();
var gwAt = new IPEndPoint(IPAddress.Parse("10.0.0.51"), 5684);
var pinned = new DtlsClientOptions();
pinned.PinnedFingerprints.Add(DtlsCertificates.Fingerprint(cert));
var secure = new DtlsClientTransport(net.Bind(), gwAt, pinned);
await secure.HandshakeAsync();
Console.WriteLine($"suite {secure.CipherSuite}, pin {DtlsCertificates.Fingerprint(cert)[..16]}…");
var gwClient = CoapClient.Create(o => o.UseDatagramTransport(() => secure).UseServer(gwAt));
await gwClient.ConnectAsync();
Console.WriteLine($"soil {(await gwClient.GetAsync("/sensors/soil")).PayloadText} %");

var other = new DtlsClientOptions();
other.PinnedFingerprints.Add(new string('0', 64));
var intruder = new DtlsClientTransport(net.Bind(), gwAt, other);
try { await intruder.HandshakeAsync(); } catch (IoTComException e) { Console.WriteLine(e.Message); }
await intruder.DisposeAsync();
await gwClient.DisposeAsync();
await gwDevice.DisposeAsync();
await gw.DisposeAsync();

## Going further
`iotcom coap serve --psk id:hex` or `--cert`, `iotcom coap get coaps://host/path --psk id:hex` (or `IOTCOM_DTLS_PSK`), `--pin <sha256>`, and the CoapsSecure sample. LwM2M uses the same `UseDtlsClient`/`UseDtlsServer`. See `docs/en/protocols/dtls.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*